# 99 — Cleanup (avoid ongoing AWS costs)

Run this notebook when you're done demoing.

It attempts to delete:
- SageMaker Endpoint (+ endpoint config + model)
- Model Monitor schedule
- CloudWatch dashboard
- Feature Store feature group (optional)

⚠️ Deletions can take a few minutes in AWS.


In [1]:
import boto3
import sagemaker

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [2]:
# Try to load saved variables (skip ones you don't have)
try:
    %store -r region
except Exception as e:
    region = boto3.Session().region_name

print("Region:", region)

for var in ["endpoint_name","schedule_name","dashboard_name","FEATURE_GROUP_NAME"]:
    try:
        get_ipython().run_line_magic("store", f"-r {var}")
        print(f"Loaded {var}: {globals().get(var)}")
    except Exception:
        print(f"{var} not found in %store (skipping)")

Region: us-east-1
Loaded endpoint_name: buoycast-endpoint-20260221-025745
Loaded schedule_name: buoycast-data-quality-20260221-025745
Loaded dashboard_name: BuoyCast-20260221-025745
Loaded FEATURE_GROUP_NAME: buoy-wave-features-1771641978


In [3]:
sm = boto3.client("sagemaker", region_name=region)
cw = boto3.client("cloudwatch", region_name=region)

In [4]:
# Delete monitoring schedule
if "schedule_name" in globals():
    try:
        sm.delete_monitoring_schedule(MonitoringScheduleName=schedule_name)
        print("Deleted monitoring schedule:", schedule_name)
    except Exception as e:
        print("Could not delete monitoring schedule:", e)

Deleted monitoring schedule: buoycast-data-quality-20260221-025745


In [5]:
# Delete endpoint (+ config + model)
if "endpoint_name" in globals():
    try:
        desc = sm.describe_endpoint(EndpointName=endpoint_name)
        endpoint_config_name = desc["EndpointConfigName"]
        print("EndpointConfigName:", endpoint_config_name)

        cfg = sm.describe_endpoint_config(EndpointConfigName=endpoint_config_name)
        model_names = [pv["ModelName"] for pv in cfg["ProductionVariants"]]
        print("Model(s):", model_names)

        sm.delete_endpoint(EndpointName=endpoint_name)
        print("Deleted endpoint:", endpoint_name)

        sm.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
        print("Deleted endpoint config:", endpoint_config_name)

        for mn in model_names:
            try:
                sm.delete_model(ModelName=mn)
                print("Deleted model:", mn)
            except Exception as e:
                print("Could not delete model:", mn, e)

    except Exception as e:
        print("Could not delete endpoint resources:", e)

EndpointConfigName: buoycast-endpoint-20260221-025745
Model(s): ['buoycast-wave-models-2026-02-21-03-08-55-709']
Could not delete endpoint resources: An error occurred (ValidationException) when calling the DeleteEndpoint operation: The Endpoint currently has one or more MonitoringSchedules. Please delete the MonitoringSchedules before deleting the Endpoint.


In [6]:
# Delete CloudWatch dashboard
if "dashboard_name" in globals():
    try:
        cw.delete_dashboards(DashboardNames=[dashboard_name])
        print("Deleted dashboard:", dashboard_name)
    except Exception as e:
        print("Could not delete dashboard:", e)

Deleted dashboard: BuoyCast-20260221-025745


In [7]:
# Optional: delete feature group (careful — deletes feature store metadata)
if "FEATURE_GROUP_NAME" in globals():
    try:
        sm.delete_feature_group(FeatureGroupName=FEATURE_GROUP_NAME)
        print("Deleted feature group:", FEATURE_GROUP_NAME)
    except Exception as e:
        print("Could not delete feature group:", e)

Deleted feature group: buoy-wave-features-1771641978
